In [ ]:
import pandas as pd
import numpy as np


อันนี้คือขั้นก่อนเอาไฟล์เข้า

In [6]:
df = pd.read_csv("diabetes.csv")
df.shape

(768, 9)

ลองเอาแถวออกก่อน

In [7]:
df.info()
df.isna().sum()
df[['Glucose','BMI']].describe()
(df[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] == 0).sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11


แถวซ้ำ

In [8]:
print(f"Shape of DataFrame before dropping duplicates: {df.shape}")
df.drop_duplicates(inplace=True)
print(f"Shape of DataFrame after dropping duplicates: {df.shape}")

Shape of DataFrame before dropping duplicates: (768, 9)
Shape of DataFrame after dropping duplicates: (768, 9)


In [9]:
df['Outcome_Label'] = df['Outcome'].map({0: 'Negative', 1: 'Positive'})
df['AgeGroup'] = pd.cut(df['Age'], bins=[20, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60+'], right=False)
df['Outcome'] = df['Outcome'].astype('category')

อันนี้ลองแก้สอดคล้อง


In [10]:
df['Outcome_Label'] = df['Outcome'].map({0: 'Negative', 1: 'Positive'})
df['AgeGroup'] = pd.cut(df['Age'], bins=[20, 30, 40, 50, 60, 100],
    labels=['20s', '30s', '40s', '50s', '60+'], right=False)
df['Outcome'] = df['Outcome'].astype('category')

จัดการค่าโดด

In [11]:
df[df['Insulin'] > 600]
df['SkinThickness'].describe()

,SkinThickness
count,768.000000
mean,20.536458
std,15.952218
min,0.000000
25%,0.000000
50%,23.000000
75%,32.000000
max,99.000000


จัดการค่าหาย

In [ ]:
# 1) ค่า 0 ในคอลัมน์เหล่านี้เป็นไปไม่ได้ทางการแพทย์ -> ถือว่าเป็นค่าหาย (NaN)
zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

print("ค่าหายก่อนเติม:")
print(df[zero_as_missing].isna().sum())
print("สัดส่วนที่หาย (%):")
print((df[zero_as_missing].isna().mean() * 100).round(1))

# 2) Insulin หายเกือบครึ่ง (374/768) -> เก็บ flag ไว้ก่อนว่าแถวไหน "มีผลตรวจจริง"
df['HasInsulinReading'] = df['Insulin'].notna().astype(int)

# 3) เติมด้วย median แยกตามกลุ่ม Outcome
#    (ใช้ median เพราะทนต่อค่าโดด และแยกกลุ่มเพื่อไม่ให้คนป่วย/ไม่ป่วยดึงค่าหากัน)
df[zero_as_missing] = (
    df.groupby('Outcome', observed=True)[zero_as_missing]
      .transform(lambda s: s.fillna(s.median()))
)

print("\nค่าหายหลังเติม:")
print(df[zero_as_missing].isna().sum())
df[zero_as_missing].describe().round(2)